# Red Pitaya Sine Wave Generator

This notebook generates a sine wave on a Red Pitaya fast analog output using SCPI over Ethernet.

Edit the parameters in the next cell, then run the cells in order.

In [ ]:
RP_IP = "169.254.121.63"
FREQUENCY_HZ = 1000.0
AMPLITUDE_V = 0.5
OFFSET_V = 0.0
CHANNEL = 1
DURATION_S = 0.0
LEAVE_ON = False

# DURATION_S = 0 means keep running until you interrupt the notebook cell.

In [ ]:
import time

try:
    import redpitaya_scpi as scpi
except ImportError:
    import scpi

In [ ]:
def validate_settings(frequency_hz, amplitude_v, offset_v, channel, duration_s):
    if frequency_hz <= 0.0:
        raise ValueError("FREQUENCY_HZ must be greater than 0")
    if not 0.0 <= amplitude_v <= 1.0:
        raise ValueError("AMPLITUDE_V must be in the range 0 to 1 V")
    if not -1.0 <= offset_v <= 1.0:
        raise ValueError("OFFSET_V must be in the range -1 to 1 V")
    if offset_v + amplitude_v > 1.0 or offset_v - amplitude_v < -1.0:
        raise ValueError("OFFSET_V +/- AMPLITUDE_V must stay within about +/-1 V")
    if channel not in (1, 2):
        raise ValueError("CHANNEL must be 1 or 2")
    if duration_s < 0.0:
        raise ValueError("DURATION_S must be >= 0")


def send(rp_s, command):
    rp_s.tx_txt(command)


def configure_sine(rp_s, channel, frequency_hz, amplitude_v, offset_v):
    prefix = f"SOUR{channel}"
    send(rp_s, f"{prefix}:FUNC SINE")
    send(rp_s, f"{prefix}:FREQ:FIX {frequency_hz}")
    send(rp_s, f"{prefix}:VOLT {amplitude_v}")
    send(rp_s, f"{prefix}:VOLT:OFFS {offset_v}")
    send(rp_s, f"OUTPUT{channel}:STATE ON")
    send(rp_s, f"{prefix}:TRig:INT")


def disable_output(rp_s, channel):
    send(rp_s, f"OUTPUT{channel}:STATE OFF")


def close_connection(rp_s):
    close = getattr(rp_s, "close", None)
    if callable(close):
        close()

In [ ]:
validate_settings(FREQUENCY_HZ, AMPLITUDE_V, OFFSET_V, CHANNEL, DURATION_S)

rp_s = scpi.scpi(RP_IP)
configure_sine(rp_s, CHANNEL, FREQUENCY_HZ, AMPLITUDE_V, OFFSET_V)

print(
    f"Generating sine wave on OUT{CHANNEL}: "
    f"frequency={FREQUENCY_HZ:g} Hz, "
    f"amplitude={AMPLITUDE_V:g} V, offset={OFFSET_V:g} V"
)

try:
    if DURATION_S > 0.0:
        time.sleep(DURATION_S)
    else:
        print("Interrupt this notebook cell to stop.")
        while True:
            time.sleep(1.0)
except KeyboardInterrupt:
    print("Stopped by user.")
finally:
    if not LEAVE_ON:
        disable_output(rp_s, CHANNEL)
        print(f"OUT{CHANNEL} disabled.")
    close_connection(rp_s)

## Manual Stop Cell

If you used `LEAVE_ON = True`, or if you want a separate stop command, run the cell below.

In [ ]:
rp_s = scpi.scpi(RP_IP)
disable_output(rp_s, CHANNEL)
close_connection(rp_s)
print(f"OUT{CHANNEL} disabled.")